# Notebook 02v3: Collect Hidden States (2/3 Depth)

Key change from v2: **Extract hidden states from 2/3 model depth**, not final layer.
This is based on the authors' OpenReview Rebuttal disclosure.

Other v2 fixes retained: enable_thinking=False, num_train/eval=500

In [ ]:
import os, sys, subprocess
REPO = 'thoughtcomm'
if os.path.exists(REPO):
    os.chdir(REPO)
    subprocess.run(['git', 'pull', 'origin', 'main'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/AUMEZAK/thoughtcomm.git'], check=True)
    os.chdir(REPO)
    subprocess.run(['pip', 'install', '-e', '.', '-q'], check=True)
# Verify v3 fixes
exec(open('tests/test_v3_fixes.py').read())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
from configs.config import ThoughtCommConfig
from models.model_utils import load_model_and_tokenizer
from data.math_data import load_math_dataset
from data.gsm8k_data import load_gsm8k_dataset
from pipeline.debate import MultiAgentDebate
from pipeline.hidden_state_collector import HiddenStateCollector
import os

config = ThoughtCommConfig.for_qwen_0_6b()
MODEL_TAG = config.model_name.replace('/', '_')
SAVE_DIR = config.save_dir
os.makedirs(SAVE_DIR, exist_ok=True)

print(f'Model: {config.model_name}')
print(f'Hidden state extraction: {config.hidden_state_layer_fraction:.1%} depth')
print(f'num_train={config.num_train}, num_eval={config.num_eval}')
print(f'n_h={config.n_h} ({config.num_agents} agents x {config.hidden_size})')

In [ ]:
# Load model
model, tokenizer = load_model_and_tokenizer(
    config.model_name, dtype=config.torch_dtype
)

# Show layer info and target extraction layer
num_layers = model.config.num_hidden_layers
target = config.target_layer_index(num_layers)
print(f'Model: {config.model_name}')
print(f'Total transformer layers: {num_layers}')
print(f'Target extraction layer: {target} ({config.hidden_state_layer_fraction:.1%} of {num_layers})')
print(f'  (Authors use 2/3 depth. Final layer would be {num_layers})')

In [ ]:
# Load datasets
math_train, math_eval = load_math_dataset(
    num_train=config.num_train, num_eval=config.num_eval, level=config.math_level
)
gsm8k_train, gsm8k_eval = load_gsm8k_dataset(
    num_train=config.num_train, num_eval=config.num_eval
)
print(f'MATH: {len(math_train)} train, {len(math_eval)} eval')
print(f'GSM8K: {len(gsm8k_train)} train, {len(gsm8k_eval)} eval')

In [ ]:
# Sanity check: Single Answer on 10 MATH problems
from pipeline.thought_comm import run_single_answer_baseline
from evaluation.math_eval import extract_boxed_answer, grade_answer

sa_sample = run_single_answer_baseline(model, tokenizer, math_eval[:10], config)
correct = 0
for resp, ex in zip(sa_sample, math_eval[:10]):
    pred = extract_boxed_answer(resp)
    if pred and grade_answer(pred, ex['answer']):
        correct += 1
print(f'Single Answer sanity check: {correct}/10 ({correct*10}%)')
print('Expected ~40-50%. If 0-5%, enable_thinking fix is NOT applied.')
print(f'Sample response (first 200 chars): {sa_sample[0][:200]}')

In [ ]:
# Compare 2/3 depth vs final layer hidden states (diagnostic)
from models.model_utils import extract_hidden_state
from utils.prompts import INITIAL_PROMPT

test_q = math_train[0]['question']
msgs = [{'role': 'user', 'content': INITIAL_PROMPT.format(question=test_q)}]
try:
    result = tokenizer.apply_chat_template(msgs, return_tensors='pt', add_generation_prompt=True, enable_thinking=False)
except TypeError:
    result = tokenizer.apply_chat_template(msgs, return_tensors='pt', add_generation_prompt=True)
if hasattr(result, 'input_ids'):
    input_ids = result['input_ids'].to(model.device)
else:
    input_ids = result.to(model.device)

h_2_3 = extract_hidden_state(model, input_ids, layer_fraction=0.667)
h_last = extract_hidden_state(model, input_ids, layer_fraction=1.0)

cos_sim = torch.nn.functional.cosine_similarity(h_2_3.unsqueeze(0), h_last.unsqueeze(0)).item()
print(f'2/3 depth: mean={h_2_3.mean():.4f}, std={h_2_3.std():.4f}, norm={h_2_3.norm():.2f}')
print(f'Last layer: mean={h_last.mean():.4f}, std={h_last.std():.4f}, norm={h_last.norm():.2f}')
print(f'Cosine similarity (2/3 vs last): {cos_sim:.4f}')
print(f'  (Low cosine = very different representations → 2/3 depth matters)')

In [ ]:
# Collect hidden states (2/3 depth) for MATH training set
debate = MultiAgentDebate(model, tokenizer, config)
collector = HiddenStateCollector(debate, config)

H_math, meta_math = collector.collect(
    math_train, save_dir=os.path.join(SAVE_DIR, f'{MODEL_TAG}_math_v3')
)
print(f'MATH hidden states: {H_math.shape}')

In [ ]:
# Collect hidden states (2/3 depth) for GSM8K training set
H_gsm8k, meta_gsm8k = collector.collect(
    gsm8k_train, save_dir=os.path.join(SAVE_DIR, f'{MODEL_TAG}_gsm8k_v3')
)
print(f'GSM8K hidden states: {H_gsm8k.shape}')

In [ ]:
# Save and verify
import json

torch.save({'H': H_math, 'metadata': meta_math},
           os.path.join(SAVE_DIR, f'{MODEL_TAG}_math_hidden_v3.pt'))
torch.save({'H': H_gsm8k, 'metadata': meta_gsm8k},
           os.path.join(SAVE_DIR, f'{MODEL_TAG}_gsm8k_hidden_v3.pt'))

summary = {
    'model': config.model_name,
    'version': 'v3',
    'hidden_state_layer_fraction': config.hidden_state_layer_fraction,
    'target_layer': config.target_layer_index(model.config.num_hidden_layers),
    'total_layers': model.config.num_hidden_layers,
    'math_h_shape': list(H_math.shape),
    'math_h_mean': H_math.float().mean().item(),
    'math_h_std': H_math.float().std().item(),
    'gsm8k_h_shape': list(H_gsm8k.shape),
    'gsm8k_h_mean': H_gsm8k.float().mean().item(),
    'gsm8k_h_std': H_gsm8k.float().std().item(),
}

os.makedirs('results', exist_ok=True)
with open(f'results/02v3_hidden_states_{MODEL_TAG}.json', 'w') as f:
    json.dump(summary, f, indent=2)

for k, v in summary.items():
    print(f'{k}: {v}')